# Call peak for 181 fragments by MACS2

In [ ]:
# Parameter settings come from signac
# shell
/home/qiaolibo/miniconda3/envs/macs2/bin/macs2 callpeak \
	    -t ./atac_fragments.tsv.gz \
	    -g 2.7e+09 \
	    -f BED \
	    --nomodel \
	    --extsize 200 \
	    --shift -100 \
	    -n $name \
	    --outdir ./summits.bed

# Harmonize cCREs by the iterative overlap peak merging procedure

In [ ]:
# Build metadata
# R
setwd('/all_summit')
list <- as.data.frame(list.files())
list$Group <- as.data.frame(t(as.data.frame(strsplit(list$`list.files()`, '_summits'))))$V1
colnames(list) <- c('Sample','Group')
list$Sample <- list$Group

write.table(list, '../metadata.txt', row.names = F, quote = F, sep = '\t')

In [ ]:
# https://github.com/corceslab/ATAC_IterativeOverlapPeakMerging
# shell
Rscript createIterativeOverlapPeakSet.R \
  --metadata metadata.txt \
  --macs2dir all_summit/ \
  --outdir merged_summit/ \
  --suffix _summits.bed \
  --blacklist hg38-blacklist.v2.bed \
  --genome hg38 \
  --spm 1 \
  --rule "(n+1)/2" \
  --extend 250

# Evaluate harmonized cCREs

## For external cCREs sets

In [ ]:
# R
library(GenomicRanges)
library(rtracklayer)
library(ChIPseeker)

cre1 = read.table('cCRE_cpeak/encode_GRCh38_cCREs_1063878.bed')
cre2 = read.table('cCRE_cpeak/sc_cCRE_GRCh38_1154611.tsv', sep = '\t')
cre3 = read.table('cCRE_cpeak/cpeaks_hg38_1567527.bed')

gr_1 = GRanges(
    seqnames = cre1$V1,
    ranges = IRanges(start = cre1$V2, end = cre1$V3),
    strand = '*'
)
gr_2 = GRanges(
    seqnames = cre2$V1,
    ranges = IRanges(start = cre2$V2, end = cre2$V3),
    strand = '*'
)
gr_3 = GRanges(
    seqnames = cre3$V1,
    ranges = IRanges(start = cre3$V2, end = cre3$V3),
    strand = '*'
)

gr1 = gr_2
gr2 = gr_3

overlaps <- findOverlaps(gr1, gr2)
length(gr1)
length(gr2)
length(overlaps)
length(unique(queryHits(overlaps)))
length(unique(subjectHits(overlaps)))
length(unique(queryHits(overlaps)))/length(gr1)
length(unique(subjectHits(overlaps)))/length(gr2)
           
jaccard_index <- sum(width(reduce(gr1[queryHits(overlaps)], min.gapwidth = 0)))/
                 sum(width(reduce(c(gr1, gr2), min.gapwidth = 0)))
jaccard_index
jaccard_index <- sum(width(reduce(gr2[subjectHits(overlaps)], min.gapwidth = 0)))/
                 sum(width(reduce(c(gr1, gr2), min.gapwidth = 0)))
jaccard_index

## For sample distribution

In [ ]:
# Python
import pandas as pd

data = pd.read_table('../merged_summit/All_Samples.fwp.filter.non_overlapping.bed', header=None)
data.iloc[:,5] = [s.rpartition('_')[0] if '_' in s else s for s in data.iloc[:,3]]
data = pd.DataFrame(data[5].value_counts())
data.to_csv('peak_sample_distribution.csv')

In [ ]:
# Python
import seaborn as sns
import matplotlib.pyplot as plt

# Plot the histogram
plt.figure(figsize=(8, 6))
ax = sns.histplot(data['count'], bins=10, kde=False, color='blue')

# Display frequencies on top of the bars
for patch in ax.patches:
    # Get the x-center and height of each bar
    x = patch.get_x() + patch.get_width() / 2
    y = patch.get_height()
    
    # Annotate the count if the height is greater than 0
    if y > 0: 
        ax.text(x, y, int(y), ha='center', va='bottom', fontsize=10)

# Set the title and labels
plt.xlabel('Count')
plt.ylabel('Frequency')
plt.title('Histogram of Count with Frequencies')

# Save as PDF
plt.savefig("histogram_with_counts_seaborn.pdf", format="pdf", bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
# The original peak number
# Python
import os

# Specify the directory path containing the files
folder_path = "../all_summit/"  # Replace with your actual directory path

# Initialize a list to store the line counts
line_counts = []

# Iterate through all files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith(".bed"):  # Filter for .bed files
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r') as f:
            # Count the number of lines in the file
            line_count = sum(1 for line in f)
            line_counts.append(line_count)

# Output the list of line counts
print("Line counts list:", line_counts)

import seaborn as sns
import matplotlib.pyplot as plt

# Example list data (assuming 'data' variable is defined)

# Set the value range for the histogram
value_range = (20000, 220000)

# Plot the histogram
plt.figure(figsize=(8, 6))
ax = sns.histplot(data, bins=10, binrange=value_range, kde=False, color='blue')

# Display sample counts on top of the bars
for patch in ax.patches:
    x = patch.get_x() + patch.get_width() / 2
    y = patch.get_height()
    
    # Only annotate if there is data in the bin (height > 0)
    if y > 0:
        ax.text(x, y, str(int(y)), ha='center', va='bottom', fontsize=10)

# Add title and labels
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.title('Histogram of List Data (Limited Range)')

# Save the plot as a PDF file
plt.savefig("histogram_with_counts_seaborn_raw.pdf", format="pdf", bbox_inches="tight")

# Display the plot
plt.show()

# Obtain ATAC peak-cell matrix by FeatureMatrix from Signac

In [ ]:
# R
library(Signac)
library(future)
plan("multicore", workers = 100)
setwd('2_cpeak_matrix/')

list <- list.files('../1_fragments/', pattern = "\\.gz$")
name <- sub("_atac.*", "", list)
peaks <- readRDS('../0_ref_peak/merged_summit/All_Samples.fwp.filter.non_overlapping.rds')

for (i in 1:length(list)){
  print(i)
  print(list[i])
  counts <- FeatureMatrix(
    fragments = CreateFragmentObject(
      path = paste0("../1_fragments/", list[i])
    ),
    features = peaks
  )
  saveRDS(counts, paste0('data/', name[i], '.rds'))
}

# cCRE conservation and enrichment analysis

In [ ]:
# conservation
library(GenomicRanges)
library(phastCons100way.UCSC.hg38)
ls("package:phastCons100way.UCSC.hg38")
phast <- phastCons100way.UCSC.hg38
phast
citation(phast)
gscores(phast, GRanges("chr7:117592326"))


peak_summit <- resize(peak, width = 1, fix = "center")

summit <- gscores(phast, peak_summit)
data <- as.data.frame(summit@elementMetadata@listData[["default"]])
colnames(data) <- 'summit'
for (i in -500:500){
  print(i)
  peak_a <- gscores(phast, shift(peak_summit, shift = i))
  data_a <- as.data.frame(peak_a@elementMetadata@listData[["default"]])
  colnames(data_a) <- paste0('summit',i)
  data <- cbind(data, data_a)
}

dat <- as.data.frame(colSums(data, na.rm = TRUE))
dat <- as.data.frame(dat[2:1002,])
colnames(dat) <- 'scores'  
dat$scores <- dat$scores/527459
dat$dis <- -500:500
plot(dat$dis, dat$scores, xlab = "summit", ylab = "PhastCons score (averaged)", pch = 16, col = "blue")

write.csv(data[,2:1002],'./processed/peak_evalu/1001_PhastCons.csv')
write.csv(dat,'./processed/peak_evalu/ava_PhastCons.csv')

data <- read.csv('./processed/peak_evalu/1001_PhastCons.csv', row.names = 1)
elments <- as.data.frame(peakAnno@anno@elementMetadata@listData[["distanceToTSS"]]>5000 | peakAnno@anno@elementMetadata@listData[["distanceToTSS"]]< -5000)
colnames(elments) <- 'elm'
table(elments$elm)

data_pro <- data[!elments$elm,]
data_dis <- data[elments$elm,]
dat_pro <- as.data.frame(colSums(data_pro, na.rm = TRUE))
dat_dis <- as.data.frame(colSums(data_dis, na.rm = TRUE))
colnames(dat_pro) <- 'scores'  
dat_pro$scores <- dat_pro$scores/(dim(data_pro)[1])
colnames(dat_dis) <- 'scores'  
dat_dis$scores <- dat_dis$scores/(dim(data_dis)[1])

colnames(dat_dis) <- 'scores_dis'
dat_dis$scores_pro <- dat_pro$scores 
dat_dis$dis <- -500:500

# random seq
# bedtools random -l 1001 -n 527459 -g  > random_regions.bed
peak <- readPeakFile('./processed/peak_evalu/random_regions.bed')
peak_summit <- resize(peak, width = 1, fix = "center")
summit <- gscores(phast, peak_summit)
data <- as.data.frame(summit@elementMetadata@listData[["default"]])
colnames(data) <- 'summit'
for (i in -500:500){
  print(i)
  peak_a <- gscores(phast, shift(peak_summit, shift = i))
  data_a <- as.data.frame(peak_a@elementMetadata@listData[["default"]])
  colnames(data_a) <- paste0('summit',i)
  data <- cbind(data, data_a)
}
write.csv(data[,2:1002],'./processed/peak_evalu/random_PhastCons.csv')
dat <- as.data.frame(colSums(data, na.rm = TRUE))
dat <- as.data.frame(dat[2:1002,])
colnames(dat) <- 'scores'  
dat$scores <- dat$scores/527459

dat_dis$scores_bg <- dat$scores

write.csv(dat_dis,'./processed/peak_evalu/all_ave_PhastCons.csv')

df <- dat_dis
# scores_pro---red
plot(df$dis, df$scores_pro, 
     type = "l",       
     xlab = "dis", 
     ylab = "score", 
     main = "Proximal vs Distal Scores", 
     col = "#e21e25", 
     lwd = 3,
     ylim = range(c(df$scores_dis, df$scores_pro)))

# scores_dis---blue
lines(df$dis, df$scores_dis, col = "#1975bc", lwd = 3)
lines(df$dis, df$scores_bg, col = "#9ea0a3", lwd = 3)

legend("topright", legend = c("scores_pro", "scores_dis"), 
       col = c("red", "#1975bc"), lty = 1)

In [ ]:
# Functional annotation and enrichment analysis
library(ChIPseeker)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
library(clusterProfiler)
setwd('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new')

peak <- readPeakFile('../0_ref_peak/merged_summit/All_Samples.fwp.filter.non_overlapping.bed')
peakAnno <- annotatePeak(peak, tssRegion=c(-3000, 3000),
                         TxDb=txdb, annoDb="org.Hs.eg.db")
plotAnnoPie(peakAnno)
plotDistToTSS(peakAnno,
              title="Distribution of transcription factor-binding loci\nrelative to TSS")


library(ReactomePA)

pathway1 <- enrichPathway(as.data.frame(peakAnno)$geneId)
head(pathway1, 2)

gene <- seq2gene(peak, tssRegion = c(-1000, 1000), flankDistance = 3000, TxDb=txdb)
pathway2 <- enrichPathway(gene)
head(pathway2, 10)

dotplot(pathway2)

# Get NucleosomeSignal and TSSEnrichment for unannotated cells

In [ ]:
# 10x_datasets
# Boukhaled_Nat_Immunol_2022
# Mimitou_Nat_Biotechnol_2021
# Swanson_Elife_2021_2 
# Wei_Immuni_2023
# Weng_Nature_2024_6
# Weng_Nature_2024_7

# R
library(Seurat)
library(Signac)
library(liger)

dataset='10x_datasets_2'
setwd(paste0('/pool1/qiaolibo/R_Project/peak_gene/',dataset,'/processed'))
pbmc<-readRDS(paste0(dataset,'.rds'))
pbmc<-readRDS('10x_datasets_1.rds')

pbmc@assays[["ATAC"]]@fragments[[1]]@path=paste0('/pool1/qiaolibo/R_Project/peak_gene/z_others/1_fragments/',dataset,'_atac_fragments.tsv.gz')
pbmc@assays[["peaks"]]@fragments[[1]]@path=paste0('/pool1/qiaolibo/R_Project/peak_gene/z_others/1_fragments/',dataset,'_atac_fragments.tsv.gz')
DefaultAssay(pbmc) <- 'ATAC'

pbmc <- NucleosomeSignal(pbmc)
pbmc <- TSSEnrichment(pbmc)
filter=pbmc@meta.data
filter$barcode=substr(rownames(filter),1,16)
filter$id<- paste0('10x_datasets',filter$barcode, dataset)
rownames(filter)=filter$id
filter=filter[,c("TSS.enrichment", "nucleosome_signal")]

saveRDS(filter,paste0('/pool1/qiaolibo/R_Project/peak_gene/z_others/4_cell_ann/',dataset,'.rds'))

In [ ]:
# merge data

# Processing 10x_datasets
library(Seurat)
library(Signac)
setwd('/pool1/qiaolibo/R_Project/peak_gene')
files <- list.files()[1:10] # List the first 10 directories (datasets)

annotation <- readRDS('z_others/used_data/annotation.rds')

# Process the first dataset
pbmc <- Read10X(paste0(files[1], '/raw/filtered_feature_bc_matrix/'))[["Peaks"]]
pbmc <- CreateSeuratObject(
  counts = CreateChromatinAssay(
    counts = pbmc,
    sep = c(":", "-"),
    # Point to the specific fragment file for this dataset
    fragments = paste0('/pool1/qiaolibo/R_Project/peak_gene/z_others/1_fragments/',
                       files[1],
                       '_atac_fragments.tsv.gz'),
  ),
  assay = "peaks"
)
Annotation(pbmc) <- annotation
pbmc <- NucleosomeSignal(object = pbmc)
pbmc <- TSSEnrichment(object = pbmc)

# Extract QC metrics and format row names
meta <- pbmc@meta.data[, c('nucleosome_signal', 'TSS.enrichment')]
rownames(meta) <- paste0(files[1], '-', rownames(meta))

# Loop through the remaining datasets (2 to 10)
for (i in 2:10){
  pbmc <- Read10X(paste0(files[i], '/raw/filtered_feature_bc_matrix/'))[["Peaks"]]
  pbmc <- CreateSeuratObject(
    counts = CreateChromatinAssay(
      counts = pbmc,
      sep = c(":", "-"),
      fragments = paste0('/pool1/qiaolibo/R_Project/peak_gene/z_others/1_fragments/',
                         files[i],
                         '_atac_fragments.tsv.gz'),
    ),
    assay = "peaks"
  )
  Annotation(pbmc) <- annotation
  pbmc <- NucleosomeSignal(object = pbmc)
  pbmc <- TSSEnrichment(object = pbmc)
  
  metaa <- pbmc@meta.data[, c('nucleosome_signal', 'TSS.enrichment')]
  rownames(metaa) <- paste0(files[i], '-', rownames(metaa))
  
  # Append to the main metadata object
  meta <- rbind(meta, metaa)
}
saveRDS(meta, 'z_others/4_cell_ann/10x_datasets_tss.rds')

# Reload the saved metadata
meta <- readRDS('z_others/4_cell_ann/10x_datasets_tss.rds')

# Summary of cell counts for different datasets:
# 10x_datasets                  12016+12012+11909+11898+10970+10691+3012+3009+2714+2711=80942
# Boukhaled_Nat_Immunol_2022     101020
# Mimitou_Nat_Biotechnol_2021     39374
# Swanson_Elife_2021_2            14680
# Wei_Immuni_2023                143804
# Weng_Nature_2024_6              23154
# Weng_Nature_2024_7               7878


# Processing Weng_Nature_2024_7
# Get all .rds files in the current directory
setwd('/pool1/qiaolibo/R_Project/peak_gene/z_others/4_cell_ann/Weng_Nature_2024_7/')
file_list <- list.files(pattern = "\\.rds$")

# Read all .rds files and store them in a list
rds_list <- lapply(file_list, readRDS)

# Merge all objects into one data frame using rbind
meta_f <- do.call(rbind, rds_list)

# Processing Weng_Nature_2024_6 (referred to as meta_g) which had issues
setwd('/pool1/qiaolibo/R_Project/peak_gene')
pbmc_1 <- readRDS('Weng_Nature_2024_6/processed/GSM8133910_Extended_Young_BMMC.rds')
pbmc_2 <- readRDS('Weng_Nature_2024_6/processed/GSM8133914_Extended_Young_HSPC.rds')

# Fix fragment path and set default assays
pbmc_1@assays[["ATAC"]]@fragments[[1]]@path <- '/pool1/qiaolibo/R_Project/peak_gene/z_others/1_fragments/Weng_Nature_2024_6-Extended_Young_BMMC_atac_fragments.tsv.gz'
DefaultAssay(pbmc_1) <- 'ATAC'
DefaultAssay(pbmc_2) <- 'ATAC'

# Calculate QC metrics
pbmc_1 <- NucleosomeSignal(object = pbmc_1)
pbmc_1 <- TSSEnrichment(object = pbmc_1)
pbmc_2 <- NucleosomeSignal(object = pbmc_2)
pbmc_2 <- TSSEnrichment(object = pbmc_2)

# Extract and format metadata for BMMC
meta_1 <- pbmc_1@meta.data[, c('nucleosome_signal', 'TSS.enrichment')]
rownames(meta_1) <- paste0('Weng_Nature_2024_6-',
                           substr(rownames(meta_1), 1, 16),
                           '-BMMC')

# Extract and format metadata for HSPC
meta_2 <- pbmc_2@meta.data[, c('nucleosome_signal', 'TSS.enrichment')]
rownames(meta_2) <- paste0('Weng_Nature_2024_6-',
                           substr(rownames(meta_2), 1, 16),
                           '-HSPC')
meta_g <- rbind(meta_1, meta_2)

# Correct naming error for Weng_Nature_2024_7
rownames(meta_f) <- paste0(substr(rownames(meta_f), 1, 18), '-',
                           substr(rownames(meta_f), 19, 40), '-1'
                           )

# Correct naming error for Wei_Immuni_2023
rownames(meta_e) <- paste0(substr(rownames(meta_e), 1, 15), '-',
                           substr(rownames(meta_e), 16, 50))

# Processing Swanson_Elife_2021_2 which had issues
name1 <- paste0('Swanson_Elife_2021_2-',
                colnames(MP0C1W1), '-MP0C1W1')
name2 <- paste0('Swanson_Elife_2021_2-',
                colnames(MP0C1W2), '-MP0C1W2')
rownames(meta_d) <- c(name1, name2)

# Merge all processed metadata
all_tss <- rbind(meta,
                 meta_b,
                 meta_c,
                 meta_d,
                 meta_e,
                 meta_f,
                 meta_g)

# Verify IDs against the "no annotation" dataset
no_ann <- read.csv('z_others/5_matrix_merge/processed/no_annotation/atac_meta.csv',
                   row.names = 1)
setequal(rownames(no_ann), rownames(all_tss))
all_tss <- all_tss[rownames(no_ann), ]

# Boukhaled_Nat_Immunol_2022 has two cells with NA values; P5/P6 have been checked
P6 <- readRDS("/pool1/qiaolibo/R_Project/peak_gene/Boukhaled_Nat_Immunol_2022/processed/P6/P6.rds")
pbmc <- P6[["P6"]]
pbmc@assays[["ATAC"]]@fragments[[1]]@path <- '/pool1/qiaolibo/R_Project/peak_gene/z_others/1_fragments/Boukhaled_Nat_Immunol_2022-P6_atac_fragments.tsv.gz'
pbmc <- NucleosomeSignal(object = pbmc)
table(is.na(pbmc@meta.data[["nucleosome_signal"]]))

# Save final merged data
write.csv(all_tss, 'z_others/4_cell_ann/no_ann_tss_merge.csv')

# Gene activity

In [ ]:
library(Signac)
library(Seurat)
library(Matrix)
setwd('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new')

cell = read.csv('./processed/gene_activity/cell.csv', row.names = 1)
sample = as.data.frame(table(cell$sample))
for (i in 57:70){
  print(i)
  cel = cell[cell$sample == sample[i,1],]
  cel$barcode = paste0(cel$barcode,'-1')
  
  count = as.data.frame(matrix(1, nrow = 1, ncol = dim(cel)[1]))
  rownames(count) = 'chr:1000-2000'
  colnames(count) = cel$barcode
  
  pbmc = CreateSeuratObject(
    counts = CreateChromatinAssay(
      counts = count,
      sep = c(":", "-"),
      fragments = paste0('../1_fragments/',sample[i,1], '_atac_fragments.tsv.gz')
    ),
    assay = "peaks"
  )
  # annotations <- readRDS('../used_data/annotation_v98.rds')
  Annotation(pbmc) <- annotations
  
  gene.activities <- GeneActivity(pbmc)
  gene.activities@Dimnames[[2]] <- rownames(cel)
  saveRDS(gene.activities, paste0('./processed/gene_activity/raw/', sample[i,1], '.rds'))
}

# Lareau_Nat_Genet_2023, two samples, provide one fragment,-1--2
# Swanson_Elife_2021, no need to add -1
# Thomson_Nat_Immunol_2024, no need to add -1
# Oguchi_Science_2024-D2_LAG3_1, one more_ before barcode
# Weng_Nature_2024, multiple samples correspond to one fragment

# merge
files <- list.files('./processed/gene_activity/raw/')
data <- readRDS(paste0('./processed/gene_activity/raw/', files[62]))
for (i in 63:100){
  print(i)
  dat <- readRDS(paste0('./processed/gene_activity/raw/', files[i]))
  data <- cbind(data,dat)
  print(dim(data)[2])
}

# gene <- as.data.frame(data@Dimnames[[1]])
# colnames(gene) <- 'gene'
# write.csv(gene, 'processed/gene_activity/gene.csv', row.names = FALSE)

cell <- as.data.frame(data@Dimnames[[2]])
colnames(cell) <- 'cell'
write.csv(cell, 'processed/gene_activity/cell_62_100_224152.csv', row.names = FALSE)

writeMM(data, 'processed/gene_activity/matrix_62_100_224152.mtx')

In [ ]:
import scanpy as sc
import pandas as pd
import anndata as ad
import os
import scanpy.external as sce
os.chdir('./processed/gene_activity/')

# ad_1 = sc.read_h5ad('./ad_1.h5ad')
# ad_2 = sc.read_h5ad('./ad_2.h5ad')
# ad_3 = sc.read_h5ad('./ad_3.h5ad')
# ad_4 = sc.read_h5ad('./ad_4.h5ad')
# ad_5 = sc.read_h5ad('./ad_5.h5ad')

# cell_1 = pd.read_csv('./cell_1_61_442373.csv')
# cell_2 = pd.read_csv('./cell_62_100_224152.csv')
# cell_3 = pd.read_csv('./cell_101_132_440751.csv')
# cell_4 = pd.read_csv('./cell_133_188_453656.csv')
# cell_5 = pd.read_csv('./cell_189_201_80934.csv')

# ad_1.var_names = cell_1['cell'].tolist()
# ad_2.var_names = cell_2['cell'].tolist()
# ad_3.var_names = cell_3['cell'].tolist()
# ad_4.var_names = cell_4['cell'].tolist()
# ad_5.var_names = cell_5['cell'].tolist()

# adatas = [ad_1, ad_2, ad_3, ad_4, ad_5]
# combined = ad.concat(
#     adatas,
#     axis=1,          
#     join="inner",     
#     fill_value=0      
# )

# combined.T.write_h5ad('./combined.h5ad')

# adata = sc.read_h5ad('./combined.h5ad')
# rna = sc.read_h5ad('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/adata.h5ad')
# adata = adata[rna.obs_names.tolist()]
# adata.obs['sample'] = rna.obs['sample'].tolist()
# adata.obs['final_ann'] = rna.obs['final_ann'].tolist()
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
# adata.write_h5ad('./norm.h5ad')
adata = sc.read_h5ad('./norm.h5ad')
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, batch_key='sample')
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')

sce.pp.harmony_integrate(adata, 'sample')
adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']

sc.tl.dendrogram(adata, "final_ann")

adata.write_h5ad('./dendrogram.h5ad')

In [ ]:
data = sc.read_h5ad('./norm.h5ad')
data

In [15]:
gene = pd.read_csv('../gene_activity/gene.csv')
gene

,gene
0,PLCXD1
1,GTPBP6
2,PPP2R3B
3,SHOX
4,CRLF2
...,...
19610,PCNT
19611,AP000471.1
19612,DIP2A
19613,S100B


In [16]:
data.var_names = gene['gene'].tolist()

In [ ]:
adata = sc.read_h5ad('./dendrogram.h5ad')
adata

In [ ]:
data.uns['dendrogram_final_ann'] = adata.uns['dendrogram_final_ann']

In [ ]:
marker_genes_dict = {
    "ILC": ["CD247", "GNLY", "NKG7"],
    "T cells": ["IL7R", "LEF1"],
    "Cycling cells": ["MKI67", "TOP2A", "CDK1"],
    "B-cell lineage": ["DNTT", "MME", "VPREB1"], 
    "HSC/MPP": [ "PROM1", "SPINK2", "CD34"],
    "Early MK": [ "ITGA2B", "ITGB3","PF4"],
    "Erythroid": [ "HBA1", "HBA2", "ALAS2"],
    "Mast cells": ["KIT", "CPA3","TPSAB1"],
    "DC": ["HLA-DRA",  "CST3", "ITGAX"],
    "Macrophages": ["RBM47", "DOCK4", "PLAUR"],
    "Monocytes": [ "VCAN", "LYZ", "S100A9"],
    "pDC": ["TCF4", "IL3RA", "CLEC4C"],
    "B cells": ["MS4A1", "CD79A"],
    "Plasma cells": ["JCHAIN", "MZB1","XBP1"]
}

In [ ]:
desired_order = ["ILC", "T cells", "Cycling cells",  "B-cell lineage", "HSC/MPP", "Early MK", "Erythroid", 
                 "Mast cells", "DC", "Macrophages", "Monocytes", "pDC", "B cells", "Plasma cells"]

In [ ]:
data.obs['final_ann'] = pd.Categorical(data.obs['final_ann'], categories=desired_order, ordered=True)

In [ ]:
sc.pl.dotplot(data, marker_genes_dict, groupby='final_ann', save='_marker_gene_activity') 

# Differential peaks

## scanpy rank_genes_groups can not compute

In [ ]:
sc.tl.rank_genes_groups(data, "l2", method="wilcoxon")
data.write('./l2.h5ad')

In [ ]:
sc.tl.rank_genes_groups(data, "l3", method="wilcoxon")
data.write('./l3.h5ad')

## snapatac2 demo

In [15]:
import snapatac2 as snap
import numpy as np
import polars as pl

snap.__version__

'2.8.0'

In [16]:
data.var_names = [s.replace("-", ":", 1) for s in data.var_names.tolist()]

In [17]:
data.var_names

Index(['chr1:792623-793123', 'chr1:794173-794673', 'chr1:804657-805157',
       'chr1:807626-808126', 'chr1:811266-811766', 'chr1:812758-813258',
       'chr1:817014-817514', 'chr1:819700-820200', 'chr1:820210-820710',
       'chr1:820751-821251',
       ...
       'chrX:155917600-155918100', 'chrX:155923064-155923564',
       'chrX:155927570-155928070', 'chrX:155945042-155945542',
       'chrX:155956046-155956546', 'chrX:155966772-155967272',
       'chrX:155968031-155968531', 'chrX:155968730-155969230',
       'chrX:155971566-155972066', 'chrX:155975921-155976421'],
      dtype='object', length=527459)

In [ ]:
# %%time
# marker_peaks = snap.tl.marker_regions(data, groupby='l2', pvalue=0.01)

In [34]:
marker_peaks

{'B cells': Index(['chr1:804657-805157', 'chr1:865563-866063', 'chr1:1032878-1033378',
        'chr1:1033535-1034035', 'chr1:1070280-1070780', 'chr1:1171654-1172154',
        'chr1:1800835-1801335', 'chr1:1833379-1833879', 'chr1:1864038-1864538',
        'chr1:1864593-1865093',
        ...
        'chrX:149916657-149917157', 'chrX:150083350-150083850',
        'chrX:150542907-150543407', 'chrX:150700250-150700750',
        'chrX:153591358-153591858', 'chrX:153742792-153743292',
        'chrX:153844624-153845124', 'chrX:153888353-153888853',
        'chrX:154819864-154820364', 'chrX:155731647-155732147'],
       dtype='object', length=5897),
 'B-cell lineage': Index(['chr1:792623-793123', 'chr1:807626-808126', 'chr1:1097677-1098177',
        'chr1:1650236-1650736', 'chr1:1711939-1712439', 'chr1:1755545-1756045',
        'chr1:1760622-1761122', 'chr1:1842213-1842713', 'chr1:1863046-1863546',
        'chr1:1934070-1934570',
        ...
        'chrX:154148684-154149184', 'chrX:154210896-1

In [ ]:
# import pickle
# data = {"name": "Alice", "age": 30}

# with open("data.pkl", "wb") as f:
#     pickle.dump(data, f)

# with open("data.pkl", "rb") as f:
#     loaded_data = pickle.load(f)

In [18]:
# import pickle

# with open("./processed/dif_peak/marker_peaks.pkl", "wb") as f:
#     pickle.dump(marker_peaks, f)

In [6]:
import pickle

with open("./processed/dif_peak/marker_peaks.pkl", "rb") as f:
    marker_peaks = pickle.load(f)

In [1]:
# import pandas as pd  

# data = marker_peaks
# for key, index_obj in data.items():
#     safe_filename = key.replace(' ', '_').replace('/', '_') + ".bed"
    
#     with open(safe_filename, 'w') as bed_file:
#         for region in index_obj:
#             chrom, pos_part = region.split(':')
#             start, end = pos_part.split('-')
#             bed_line = f"{chrom}\t{start}\t{end}\n"
#             bed_file.write(bed_line)

In [ ]:
snap.pl.regions(data, groupby='l2', peaks=marker_peaks, interactive=False, width= 600, height=800, out_file='./figures/dif_peak.pdf')

In [10]:
%%time
motifs = snap.tl.motif_enrichment(
    motifs=snap.datasets.cis_bp(unique=True),
    regions=marker_peaks,
    genome_fasta=snap.genome.hg38,
)

2025-02-26 14:53:50 - INFO - Fetching 156915 sequences ...
2025-02-26 14:54:03 - INFO - Computing enrichment ...
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1165/1165 [14:01<00:00,  1.38it/s]

CPU times: user 6h 21min 19s, sys: 36min 57s, total: 6h 58min 17s
Wall time: 14min 14s


In [11]:
motifs

{'B cells': shape: (1_165, 6)
 ┌────────────────────────┬────────────┬────────┬───────────────────┬──────────┬──────────────────┐
 │ id                     ┆ name       ┆ family ┆ log2(fold change) ┆ p-value  ┆ adjusted p-value │
 │ ---                    ┆ ---        ┆ ---    ┆ ---               ┆ ---      ┆ ---              │
 │ str                    ┆ str        ┆ null   ┆ f64               ┆ f64      ┆ f64              │
 ╞════════════════════════╪════════════╪════════╪═══════════════════╪══════════╪══════════════════╡
 │ AC023509.3+M02872_2.00 ┆ AC023509.3 ┆ null   ┆ 0.294747          ┆ 0.178755 ┆ 0.306701         │
 │ AC138696.1+M04597_2.00 ┆ AC138696.1 ┆ null   ┆ -0.406522         ┆ 0.041444 ┆ 0.130045         │
 │ AHR+M09817_2.00        ┆ AHR        ┆ null   ┆ 0.255811          ┆ 0.199716 ┆ 0.316165         │
 │ AIRE+M09375_2.00       ┆ AIRE       ┆ null   ┆ 0.002916          ┆ 0.465052 ┆ 0.493147         │
 │ ALX1+M05327_2.00       ┆ ALX1       ┆ null   ┆ 0.428238          ┆ 

In [12]:
import pickle

with open("./processed/dif_peak/motifs.pkl", "wb") as f:
    pickle.dump(motifs, f)

In [ ]:
snap.pl.motif_enrichment(motifs, max_fdr=0.0001, min_log_fc= 2.5,height=1600, interactive=False, out_file='./figures/motif.pdf')

## Statistical peak distribution in the sample

In [3]:
import os
import pandas as pd

folder_path = './processed/dif_peak/sample/'  

txt_files = [f for f in os.listdir(folder_path) if f.endswith('.bed')]
txt_files

['chen_nat_methods_2022-lane1.atac.bed',
 'chen_nat_methods_2022-lane2.atac.bed',
 'cheong_cell_2023_1-hdu.atac.bed',
 'cheong_cell_2023_1-hda.atac.bed',
 'cheong_cell_2023_1-hdv.atac.bed',
 'cheong_cell_2023_1-hdj.atac.bed',
 'cheong_cell_2023_1-hd_ad.atac.bed',
 'cheong_cell_2023_1-jcov101.atac.bed',
 'cheong_cell_2023_1-jcov101_2.atac.bed',
 'cheong_cell_2023_1-jcov110.atac.bed',
 'cheong_cell_2023_1-jcov111.atac.bed',
 'cheong_cell_2023_1-jcov114.atac.bed',
 'cheong_cell_2023_1-jcov115.atac.bed',
 'cheong_cell_2023_1-jcov117.atac.bed',
 'cheong_cell_2023_1-jcov119.atac.bed',
 'cheong_cell_2023_1-jcov122.atac.bed',
 'cheong_cell_2023_1-jcov124.atac.bed',
 'cheong_cell_2023_1-jcov127.atac.bed',
 'cheong_cell_2023_1-jcov21.atac.bed',
 'cheong_cell_2023_1-jcov42.atac.bed',
 'cheong_cell_2023_1-jcov42_20.atac.bed',
 'cheong_cell_2023_1-jcov49_2.atac.bed',
 'cheong_cell_2023_1-jcov52.atac.bed',
 'cheong_cell_2023_1-jcov56_9.atac.bed',
 'cheong_cell_2023_1-jcov60_6.atac.bed',
 'cheong_cel

In [18]:
combined_df = pd.DataFrame()

for txt_file in txt_files:
    file_path = os.path.join(folder_path, txt_file)
    col_name = os.path.splitext(txt_file)[0]
    df = pd.read_csv(file_path, header=None, sep='\t')
    combined_df[txt_file] = df[3].tolist()

/tmp/ipykernel_3473040/113298402.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined_df[txt_file] = df[3].tolist()
/tmp/ipykernel_3473040/113298402.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined_df[txt_file] = df[3].tolist()
/tmp/ipykernel_3473040/113298402.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented fram

In [19]:
combined_df

,chen_nat_methods_2022-lane1.atac.bed,chen_nat_methods_2022-lane2.atac.bed,cheong_cell_2023_1-hdu.atac.bed,cheong_cell_2023_1-hda.atac.bed,cheong_cell_2023_1-hdv.atac.bed,cheong_cell_2023_1-hdj.atac.bed,cheong_cell_2023_1-hd_ad.atac.bed,cheong_cell_2023_1-jcov101.atac.bed,cheong_cell_2023_1-jcov101_2.atac.bed,cheong_cell_2023_1-jcov110.atac.bed,...,marderstein_nature_2024-ts21_11_f.atac.bed,marderstein_nature_2024-ts21_11_g.atac.bed,marderstein_nature_2024-ts21_16_a.atac.bed,marderstein_nature_2024-ts21_16_b.atac.bed,marderstein_nature_2024-h_5_a.atac.bed,marderstein_nature_2024-h_5_b.atac.bed,marderstein_nature_2024-h_6_a.atac.bed,marderstein_nature_2024-h_7_a.atac.bed,marderstein_nature_2024-h_7_b.atac.bed,marderstein_nature_2024-h_7_c.atac.bed
0,0.000000,0.000000,0.000000,0.001137,0.001024,0.000000,0.000000,0.000000,0.001895,0.001264,...,0.001120,0.002242,0.000872,0.001652,0.003544,0.001951,0.001449,0.001685,0.000000,0.000942
1,0.000000,0.000000,0.004473,0.002270,0.001024,0.002245,0.004221,0.003930,0.000000,0.001264,...,0.003352,0.004471,0.004343,0.004283,0.002955,0.005345,0.003375,0.003363,0.003110,0.002820
2,0.000000,0.000000,0.000000,0.000000,0.004079,0.000000,0.005617,0.000000,0.000000,0.001264,...,0.001120,0.002242,0.000872,0.003627,0.002365,0.005345,0.001932,0.001685,0.003110,0.001882
3,0.000000,0.000000,0.000000,0.000000,0.001024,0.001124,0.001412,0.000000,0.003779,0.000000,...,0.001493,0.002242,0.000872,0.000992,0.001184,0.001464,0.001932,0.003363,0.004657,0.000000
4,0.000000,0.000000,0.000000,0.001137,0.000000,0.002245,0.000000,0.000000,0.001895,0.000000,...,0.001120,0.001794,0.000436,0.001322,0.000593,0.000977,0.001932,0.003363,0.003110,0.002820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527454,0.028608,0.030733,0.030035,0.046801,0.037991,0.044506,0.055137,0.051207,0.057654,0.096776,...,0.098754,0.118205,0.081649,0.087493,0.089649,0.088794,0.098299,0.102631,0.096888,0.095310
527455,0.000000,0.000000,0.008882,0.040778,0.011111,0.015435,0.024636,0.032016,0.085498,0.053100,...,0.092810,0.108268,0.091190,0.089918,0.062014,0.061117,0.070828,0.045781,0.044606,0.046644
527456,0.000000,0.002299,0.008882,0.000000,0.004079,0.001124,0.007009,0.005878,0.013051,0.006278,...,0.018646,0.020610,0.028253,0.021318,0.018021,0.018669,0.019896,0.014947,0.019853,0.023896
527457,0.000000,0.000000,0.000000,0.004525,0.005091,0.005587,0.004221,0.011654,0.003779,0.011227,...,0.026790,0.035736,0.017101,0.024145,0.025933,0.030238,0.029082,0.023049,0.013833,0.027468


In [26]:
combined_df.index = df[0]+':'+df[1].astype('str')+'-'+df[2].astype('str')

In [27]:
combined_df

,chen_nat_methods_2022-lane1.atac.bed,chen_nat_methods_2022-lane2.atac.bed,cheong_cell_2023_1-hdu.atac.bed,cheong_cell_2023_1-hda.atac.bed,cheong_cell_2023_1-hdv.atac.bed,cheong_cell_2023_1-hdj.atac.bed,cheong_cell_2023_1-hd_ad.atac.bed,cheong_cell_2023_1-jcov101.atac.bed,cheong_cell_2023_1-jcov101_2.atac.bed,cheong_cell_2023_1-jcov110.atac.bed,...,marderstein_nature_2024-ts21_11_f.atac.bed,marderstein_nature_2024-ts21_11_g.atac.bed,marderstein_nature_2024-ts21_16_a.atac.bed,marderstein_nature_2024-ts21_16_b.atac.bed,marderstein_nature_2024-h_5_a.atac.bed,marderstein_nature_2024-h_5_b.atac.bed,marderstein_nature_2024-h_6_a.atac.bed,marderstein_nature_2024-h_7_a.atac.bed,marderstein_nature_2024-h_7_b.atac.bed,marderstein_nature_2024-h_7_c.atac.bed
chr1:792623-793123,0.000000,0.000000,0.000000,0.001137,0.001024,0.000000,0.000000,0.000000,0.001895,0.001264,...,0.001120,0.002242,0.000872,0.001652,0.003544,0.001951,0.001449,0.001685,0.000000,0.000942
chr1:794173-794673,0.000000,0.000000,0.004473,0.002270,0.001024,0.002245,0.004221,0.003930,0.000000,0.001264,...,0.003352,0.004471,0.004343,0.004283,0.002955,0.005345,0.003375,0.003363,0.003110,0.002820
chr1:804657-805157,0.000000,0.000000,0.000000,0.000000,0.004079,0.000000,0.005617,0.000000,0.000000,0.001264,...,0.001120,0.002242,0.000872,0.003627,0.002365,0.005345,0.001932,0.001685,0.003110,0.001882
chr1:807626-808126,0.000000,0.000000,0.000000,0.000000,0.001024,0.001124,0.001412,0.000000,0.003779,0.000000,...,0.001493,0.002242,0.000872,0.000992,0.001184,0.001464,0.001932,0.003363,0.004657,0.000000
chr1:811266-811766,0.000000,0.000000,0.000000,0.001137,0.000000,0.002245,0.000000,0.000000,0.001895,0.000000,...,0.001120,0.001794,0.000436,0.001322,0.000593,0.000977,0.001932,0.003363,0.003110,0.002820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:155966772-155967272,0.028608,0.030733,0.030035,0.046801,0.037991,0.044506,0.055137,0.051207,0.057654,0.096776,...,0.098754,0.118205,0.081649,0.087493,0.089649,0.088794,0.098299,0.102631,0.096888,0.095310
chrX:155968031-155968531,0.000000,0.000000,0.008882,0.040778,0.011111,0.015435,0.024636,0.032016,0.085498,0.053100,...,0.092810,0.108268,0.091190,0.089918,0.062014,0.061117,0.070828,0.045781,0.044606,0.046644
chrX:155968730-155969230,0.000000,0.002299,0.008882,0.000000,0.004079,0.001124,0.007009,0.005878,0.013051,0.006278,...,0.018646,0.020610,0.028253,0.021318,0.018021,0.018669,0.019896,0.014947,0.019853,0.023896
chrX:155971566-155972066,0.000000,0.000000,0.000000,0.004525,0.005091,0.005587,0.004221,0.011654,0.003779,0.011227,...,0.026790,0.035736,0.017101,0.024145,0.025933,0.030238,0.029082,0.023049,0.013833,0.027468


In [28]:
combined_df.to_csv('./processed/dif_peak/sample_combined_df.txt', sep='\t', index=False)

## Peak statistics

In [29]:
non_zero_counts = (combined_df != 0).sum(axis=1)
non_zero_counts

chr1:792623-793123          150
chr1:794173-794673          178
chr1:804657-805157          169
chr1:807626-808126          129
chr1:811266-811766          148
                           ... 
chrX:155966772-155967272    201
chrX:155968031-155968531    199
chrX:155968730-155969230    193
chrX:155971566-155972066    189
chrX:155975921-155976421    192
Length: 527459, dtype: int64

In [32]:
bins = [1, 20, 40, 60, 80, 100, 120, 140, 160, 180, 201]  
labels = ['2-20', '21-40', '41-60', '61-80', '81-100', '101-120', '121-140', '141-160', '161-180', '181-201']

grouped_counts = pd.cut(non_zero_counts, bins=bins, labels=labels, include_lowest=True).value_counts().reindex(labels, fill_value=0)

In [33]:
grouped_counts

2-20            2
21-40          16
41-60          46
61-80          94
81-100        188
101-120       499
121-140      2432
141-160     16180
161-180     60871
181-201    447131
Name: count, dtype: int64

In [ ]:
sizes = [845, 2432, 16180, 60871, 447131]
labels = ['2-120', '121-140', '141-160', '161-180', '181-201']
colors = ['#FFCC00', '#FF9999', '#66B2FF', '#99FF99', '#FFCC99'] 

explode = (0, 0, 0, 0, 0) 

plt.figure(figsize=(8, 8))
plt.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',  startangle=90)
plt.title('Pie Chart of Non-Zero Counts Distribution')
plt.axis('equal')  

plt.savefig('non_zero_counts_pie_chart.pdf')
plt.show()

In [ ]:
sizes = [2, 16, 46, 94, 188, 499]
labels = ['2-20', '21-40', '41-60', '61-80 adoption', '81-100', '101-120']
colors = ['#FFCC00', '#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#CC99FF']  
explode = (0, 0, 0, 0, 0, 0)  

plt.figure(figsize=(8, 8))
plt.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Pie Chart of Non-Zero Counts Distribution (2-120)')
plt.axis('equal') 

plt.savefig('non_zero_counts_pie_chart_2-120.pdf')
plt.show()

# Sample statistics

In [46]:
non_zero_counts = (combined_df != 0).sum(axis=0)
non_zero_counts

chen_nat_methods_2022-lane1.atac.bed      307794
chen_nat_methods_2022-lane2.atac.bed      289351
cheong_cell_2023_1-hdu.atac.bed           346763
cheong_cell_2023_1-hda.atac.bed           503431
cheong_cell_2023_1-hdv.atac.bed           507279
                                           ...  
marderstein_nature_2024-h_5_b.atac.bed    527114
marderstein_nature_2024-h_6_a.atac.bed    526913
marderstein_nature_2024-h_7_a.atac.bed    526191
marderstein_nature_2024-h_7_b.atac.bed    522555
marderstein_nature_2024-h_7_c.atac.bed    526156
Length: 201, dtype: int64

In [50]:
non_zero_counts.min()

289351

In [51]:
non_zero_counts.max()

527326

In [53]:
28000,53000

(28000, 53000)

In [54]:
53000-28000

25000

In [2]:
import matplotlib.pyplot as plt
import numpy as np

range_min, range_max = 280000, 530000
bins = np.linspace(range_min, range_max, 11)  


plt.figure(figsize=(10, 6))
n, bins, patches = plt.hist(non_zero_counts, bins=bins, range=(range_min, range_max), color='skyblue', edgecolor='black')

for i in range(len(n)):
    plt.text(bins[i] + (bins[i+1] - bins[i]) / 2,  
             n[i] + max(n) * 0.01,                  
             int(n[i]),                             
             ha='center', va='bottom', fontsize=10)

plt.title('Histogram of Non-Zero Counts (28000-53000)')
plt.xlabel('Number of Non-Zero Elements')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.xticks(bins, rotation=45)

plt.savefig('non_zero_counts_histogram_280000-530000_with_counts.pdf')
plt.show()

print('non_zero_counts_histogram_28000-53000_with_counts.png')

NameError: name 'non_zero_counts' is not defined

<Figure size 1000x600 with 0 Axes>